# Edge cases of the dependency tree visualiser

A visual check of `DependencyTreeVisualiser` on inputs that stress the layout: very
long sentences, texts with many sentences, tiny sentences, non-projective and
unusual trees, long labels and words, special characters, overlapping boxes and
highlights at the edges of a sentence. `demo.ipynb` shows the normal usage.

Real parser output comes from `maltparser_syntax` (needs Java). Trees with a
specific shape are built by hand with `make_text` below.

In [1]:
from estnltk import Layer, Text

from estnltk_syntax_vis import DependencyTreeVisualiser, Style

In [2]:
def make_text(raw, trees, layer="syntax"):
    """`Text` with words, sentences and a hand-made syntax layer.

    `trees` gives `(heads, deprels)` per sentence; heads are 1-based, 0 = root,
    as in CoNLL-U. `deprels` may be a single string used for every word.
    """
    text = Text(raw).tag_layer(["words", "sentences"])
    syntax = Layer(
        layer, attributes=("id", "head", "deprel"), text_object=text, parent="words"
    )
    for sentence, (heads, deprels) in zip(text.sentences, trees, strict=True):
        words = [w for w in text.words if sentence.start <= w.start < sentence.end]
        if isinstance(deprels, str):
            deprels = [deprels] * len(words)
        assert len(words) == len(heads) == len(deprels), [w.text for w in words]
        for i, (word, head, deprel) in enumerate(
            zip(words, heads, deprels, strict=True), 1
        ):
            syntax.add_annotation(word, id=i, head=head, deprel=deprel)
    text.add_layer(syntax)
    return text


hand = DependencyTreeVisualiser(syntax_layer="syntax")

## Very long sentences

A long sentence becomes a wide diagram with many arc levels. Each diagram is in a
horizontally scrollable `div`, so the notebook stays usable.

In [3]:
long_text = Text(
    "Kui Eesti Vabariigi valitsus otsustas eelmise aasta sügisel, et Tallinna ja "
    "Tartu vahelise raudtee kiirendamiseks ehitatakse uus elektrifitseeritud "
    "rööbastee, mille valmimist oodati juba 2028. aastaks, ei osanud keegi arvata, "
    "et projekti eelarve kasvab ehitusmaterjalide hinnatõusu, tööjõupuuduse ning "
    "keeruliste pinnasetingimuste tõttu peaaegu kaks korda suuremaks, kui algselt "
    "planeeritud oli."
)
long_text.tag_layer(["maltparser_syntax", "ner"])
print(len(long_text.sentences), "sentence(s),", len(long_text.words), "words")

1 sentence(s), 55 words


In [4]:
malt = DependencyTreeVisualiser(syntax_layer="maltparser_syntax")
malt(long_text)

root 
 mark 
 nmod 
 nmod 
 nsubj 
 amod 
 nmod 
 obl 
 punct 
 mark 
 obl 
 cc 
 conj 
 amod 
 nmod 
 obl 
 ccomp 
 amod 
 acl 
 obj 
 punct 
 nmod 
 obj 
 acl:relcl 
 advmod 
 amod 
 obl 
 punct 
 aux 
 advcl 
 nsubj 
 xcomp 
 punct 
 mark 
 nmod 
 nsubj 
 ccomp 
 nmod 
 obl 
 punct 
 conj 
 cc 
 amod 
 conj 
 case 
 advmod 
 nummod 
 obl 
 xcomp 
 punct 
 mark 
 advmod 
 advcl 
 aux 
 punct 
 Kui 
 Eesti 
 Vabariigi 
 valitsus 
 otsustas 
 eelmise 
 aasta 
 sügisel 
 , 
 et 
 Tallinna 
 ja 
 Tartu 
 vahelise 
 raudtee 
 kiirendamiseks 
 ehitatakse 
 uus 
 elektrifitseeritud 
 rööbastee 
 , 
 mille 
 valmimist 
 oodati 
 juba 
 2028. 
 aastaks 
 , 
 ei 
 osanud 
 keegi 
 arvata 
 , 
 et 
 projekti 
 eelarve 
 kasvab 
 ehitusmaterjalide 
 hinnatõusu 
 , 
 tööjõupuuduse 
 ning 
 keeruliste 
 pinnasetingimuste 
 tõttu 
 peaaegu 
 kaks 
 korda 
 suuremaks 
 , 
 kui 
 algselt 
 planeeritud 
 oli 
 .

With annotation rows and NE boxes the diagram also grows downwards.

In [5]:
malt_full = DependencyTreeVisualiser(
    syntax_layer="maltparser_syntax",
    rows=[
        ("morph_analysis", "lemma"),
        ("morph_analysis", "partofspeech"),
        ("morph_analysis", "form"),
    ],
    span_layer="ner",
    span_label="nertag",
)
malt_full(long_text)

root 
 mark 
 nmod 
 nmod 
 nsubj 
 amod 
 nmod 
 obl 
 punct 
 mark 
 obl 
 cc 
 conj 
 amod 
 nmod 
 obl 
 ccomp 
 amod 
 acl 
 obj 
 punct 
 nmod 
 obj 
 acl:relcl 
 advmod 
 amod 
 obl 
 punct 
 aux 
 advcl 
 nsubj 
 xcomp 
 punct 
 mark 
 nmod 
 nsubj 
 ccomp 
 nmod 
 obl 
 punct 
 conj 
 cc 
 amod 
 conj 
 case 
 advmod 
 nummod 
 obl 
 xcomp 
 punct 
 mark 
 advmod 
 advcl 
 aux 
 punct 
 Kui 
 kui 
 J 
 
 Eesti 
 Eesti 
 H 
 sg g 
 Vabariigi 
 vabariik 
 S 
 sg g 
 valitsus 
 valitsus 
 S 
 sg n 
 otsustas 
 otsustama 
 V 
 s 
 eelmise 
 eelmine 
 A 
 sg g 
 aasta 
 aasta 
 S 
 sg g 
 sügisel 
 sügis 
 S 
 sg ad 
 , 
 , 
 Z 
 
 et 
 et 
 J 
 
 Tallinna 
 Tallinn 
 H 
 sg g 
 ja 
 ja 
 J 
 
 Tartu 
 Tartu 
 H 
 sg g 
 vahelise 
 vaheline 
 A 
 sg g 
 raudtee 
 raudtee 
 S 
 sg g 
 kiirendamiseks 
 kiirendamine 
 S 
 sg tr 
 ehitatakse 
 ehitama 
 V 
 takse 
 uus 
 uus 
 A 
 sg n 
 elektrifitseeritud 
 elektrifitseerima 
 V 
 tud 
 rööbastee 
 rööbastee 
 S 
 sg n 
 , 
 , 
 Z 
 
 mille 
 mis 
 P 
 sg g 
 valmimist 
 valmimine 
 S 
 sg p 
 oodati 
 ootama 
 V 
 ti 
 juba 
 juba 
 D 
 
 2028. 
 2028. 
 O 
 ? 
 aastaks 
 aasta 
 S 
 sg tr 
 , 
 , 
 Z 
 
 ei 
 ei 
 V 
 neg 
 osanud 
 oskama 
 V 
 nud 
 keegi 
 keegi 
 P 
 sg n 
 arvata 
 arvama 
 V 
 da 
 , 
 , 
 Z 
 
 et 
 et 
 J 
 
 projekti 
 projekt 
 S 
 sg g 
 eelarve 
 eelarve 
 S 
 sg g 
 kasvab 
 kasvama 
 V 
 b 
 ehitusmaterjalide 
 ehitusmaterjal 
 S 
 pl g 
 hinnatõusu 
 hinnatõus 
 S 
 sg g 
 , 
 , 
 Z 
 
 tööjõupuuduse 
 tööjõupuudus 
 S 
 sg g 
 ning 
 ning 
 J 
 
 keeruliste 
 keeruline 
 A 
 pl g 
 pinnasetingimuste 
 pinnasetingimus 
 S 
 pl g 
 tõttu 
 tõttu 
 K 
 
 peaaegu 
 peaaegu 
 D 
 
 kaks 
 kaks 
 N 
 sg n 
 korda 
 kord 
 S 
 sg p 
 suuremaks 
 suurem 
 C 
 sg tr 
 , 
 , 
 Z 
 
 kui 
 kui 
 J 
 
 algselt 
 algselt 
 D 
 
 planeeritud 
 planeerima 
 V 
 tud 
 oli 
 olema 
 V 
 s 
 . 
 . 
 Z 
 
 LOC 
 LOC 
 LOC 
 
 lemma 
 partofspeech 
 form

Highlights at the start and end of a long sentence, with and without a window. The
window keeps the fragments readable even when the heads are far away.

In [6]:
first, last = long_text.words[0], long_text.words[-2]
malt(long_text, highlight=[first, last])

root 
 nmod 
 nmod 
 nsubj 
 amod 
 nmod 
 obl 
 punct 
 mark 
 obl 
 cc 
 conj 
 amod 
 nmod 
 obl 
 ccomp 
 amod 
 acl 
 obj 
 punct 
 nmod 
 obj 
 acl:relcl 
 advmod 
 amod 
 obl 
 punct 
 aux 
 advcl 
 nsubj 
 xcomp 
 punct 
 mark 
 nmod 
 nsubj 
 ccomp 
 nmod 
 obl 
 punct 
 conj 
 cc 
 amod 
 conj 
 case 
 advmod 
 nummod 
 obl 
 xcomp 
 punct 
 mark 
 advmod 
 advcl 
 punct 
 mark 
 aux 
 Kui 
 Eesti 
 Vabariigi 
 valitsus 
 otsustas 
 eelmise 
 aasta 
 sügisel 
 , 
 et 
 Tallinna 
 ja 
 Tartu 
 vahelise 
 raudtee 
 kiirendamiseks 
 ehitatakse 
 uus 
 elektrifitseeritud 
 rööbastee 
 , 
 mille 
 valmimist 
 oodati 
 juba 
 2028. 
 aastaks 
 , 
 ei 
 osanud 
 keegi 
 arvata 
 , 
 et 
 projekti 
 eelarve 
 kasvab 
 ehitusmaterjalide 
 hinnatõusu 
 , 
 tööjõupuuduse 
 ning 
 keeruliste 
 pinnasetingimuste 
 tõttu 
 peaaegu 
 kaks 
 korda 
 suuremaks 
 , 
 kui 
 algselt 
 planeeritud 
 oli 
 .

In [7]:
malt(long_text, highlight=[first, last], window=0)

root 
 mark 
 aux 
 Kui 
 … 
 otsustas 
 … 
 planeeritud 
 oli 
 …

In [8]:
malt(long_text, highlight=[first, last], window=2)

root 
 nmod 
 advmod 
 punct 
 mark 
 aux 
 Kui 
 Eesti 
 Vabariigi 
 … 
 otsustas 
 … 
 algselt 
 planeeritud 
 oli 
 .

An enumeration: many dependents of one word give it many legs, and the conjuncts
nest above each other.

In [9]:
list_text = Text(
    "Poes müüdi õunu, pirne, ploome, kirsse, apelsine, banaane, viinamarju, "
    "mandariine, ananasse, kiivisid, mangosid, greipe, sidruneid ja laime."
)
list_text.tag_layer(["maltparser_syntax"])
malt(list_text)

root 
 obl 
 obj 
 punct 
 conj 
 punct 
 conj 
 punct 
 conj 
 punct 
 conj 
 punct 
 conj 
 punct 
 conj 
 punct 
 conj 
 punct 
 conj 
 punct 
 conj 
 punct 
 conj 
 punct 
 conj 
 punct 
 conj 
 cc 
 conj 
 punct 
 Poes 
 müüdi 
 õunu 
 , 
 pirne 
 , 
 ploome 
 , 
 kirsse 
 , 
 apelsine 
 , 
 banaane 
 , 
 viinamarju 
 , 
 mandariine 
 , 
 ananasse 
 , 
 kiivisid 
 , 
 mangosid 
 , 
 greipe 
 , 
 sidruneid 
 ja 
 laime 
 .

Hand-made extremes: every word depends on the first one (one hub with 25 legs and
25 levels), and a chain where each word depends on the next one, which stays flat
because arcs that only share an end word share a level.

In [10]:
words = " ".join(f"s{i}" for i in range(1, 27))
hub = make_text(words, [([0] + [1] * 25, "dep")])
hand(hub)

dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 s1 
 s2 
 s3 
 s4 
 s5 
 s6 
 s7 
 s8 
 s9 
 s10 
 s11 
 s12 
 s13 
 s14 
 s15 
 s16 
 s17 
 s18 
 s19 
 s20 
 s21 
 s22 
 s23 
 s24 
 s25 
 s26

In [11]:
chain_heads = list(range(2, 27)) + [0]
chain = make_text(words, [(chain_heads, "dep")])
hand(chain)

dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 s1 
 s2 
 s3 
 s4 
 s5 
 s6 
 s7 
 s8 
 s9 
 s10 
 s11 
 s12 
 s13 
 s14 
 s15 
 s16 
 s17 
 s18 
 s19 
 s20 
 s21 
 s22 
 s23 
 s24 
 s25 
 s26

Nested arcs: word *i* depends on word *n + 1 − i* in the first half, so every arc
contains the next one.

In [12]:
n = 20
nested_heads = [
    n + 1 - i if i <= n // 2 else 0 if i == n // 2 + 1 else i - 1
    for i in range(1, n + 1)
]
nested = make_text(" ".join(f"w{i}" for i in range(1, n + 1)), [(nested_heads, "dep")])
hand(nested)

dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 w1 
 w2 
 w3 
 w4 
 w5 
 w6 
 w7 
 w8 
 w9 
 w10 
 w11 
 w12 
 w13 
 w14 
 w15 
 w16 
 w17 
 w18 
 w19 
 w20

## Texts with several sentences

Each sentence gets its own diagram, including one-word sentences, questions,
exclamations and quoted speech.

In [13]:
multi = Text(
    "Tere! Kas sa tulid eile koosolekule? Jah. "
    '"Ma ei tea, kas see on hea mõte," ütles Mari Tamm Tartu Ülikoolist. '
    "Pärast seda läksime kõik koju. Aitäh!"
)
multi.tag_layer(["maltparser_syntax", "ner"])
for i, s in enumerate(multi.sentences):
    print(i, repr(s.enclosing_text))

0 'Tere!'
1 'Kas sa tulid eile koosolekule?'
2 'Jah.'
3 '"Ma ei tea, kas see on hea mõte," ütles Mari Tamm Tartu Ülikoolist.'
4 'Pärast seda läksime kõik koju.'
5 'Aitäh!'


In [14]:
malt_ner = DependencyTreeVisualiser(
    syntax_layer="maltparser_syntax",
    rows=[("morph_analysis", "partofspeech")],
    span_layer="ner",
    span_label="nertag",
)
malt_ner(multi)

root 
 punct 
 Tere 
 I 
 ! 
 Z 
 
 partofspeech 
 
 
 
 
 
 
 
 
 
 
 root 
 advmod 
 nsubj 
 advmod 
 obl 
 punct 
 Kas 
 D 
 sa 
 P 
 tulid 
 V 
 eile 
 D 
 koosolekule 
 S 
 ? 
 Z 
 
 partofspeech 
 
 
 
 
 
 
 root 
 punct 
 Jah 
 D 
 . 
 Z 
 
 partofspeech 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 root 
 punct 
 nsubj 
 aux 
 punct 
 advmod 
 nsubj:cop 
 cop 
 amod 
 ccomp 
 punct 
 punct 
 parataxis 
 nsubj 
 flat 
 nmod 
 obl 
 punct 
 " 
 Z 
 Ma 
 P 
 ei 
 V 
 tea 
 V 
 , 
 Z 
 kas 
 D 
 see 
 P 
 on 
 V 
 hea 
 A 
 mõte 
 S 
 , 
 Z 
 " 
 Z 
 ütles 
 V 
 Mari 
 H 
 Tamm 
 H 
 Tartu 
 H 
 Ülikoolist 
 S 
 . 
 Z 
 PER 
 ORG 
 
 partofspeech 
 
 
 
 
 
 
 
 
 
 
 root 
 case 
 obl 
 det 
 obl 
 punct 
 Pärast 
 K 
 seda 
 P 
 läksime 
 V 
 kõik 
 P 
 koju 
 S 
 . 
 Z 
 
 partofspeech 
 
 
 
 
 
 
 root 
 punct 
 Aitäh 
 I 
 ! 
 Z 
 
 partofspeech

Selecting one sentence: by index (negative indices count from the end), or through
a highlight, which finds its sentence by position.

In [15]:
malt_ner(multi, sentence=-1)

root 
 punct 
 Aitäh 
 I 
 ! 
 Z 
 
 partofspeech

In [16]:
malt_ner(multi, highlight=multi.ner[-1])

root 
 punct 
 nsubj 
 aux 
 punct 
 advmod 
 nsubj:cop 
 cop 
 amod 
 ccomp 
 punct 
 punct 
 parataxis 
 nsubj 
 flat 
 punct 
 obl 
 nmod 
 " 
 Z 
 Ma 
 P 
 ei 
 V 
 tea 
 V 
 , 
 Z 
 kas 
 D 
 see 
 P 
 on 
 V 
 hea 
 A 
 mõte 
 S 
 , 
 Z 
 " 
 Z 
 ütles 
 V 
 Mari 
 H 
 Tamm 
 H 
 Tartu 
 H 
 Ülikoolist 
 S 
 . 
 Z 
 PER 
 ORG 
 
 partofspeech

In [17]:
malt_ner(multi, sentence=3, highlight=[0, 1, 2], window=1)

root 
 punct 
 nsubj 
 aux 
 " 
 Z 
 Ma 
 P 
 ei 
 V 
 tea 
 V 
 … 
 
 partofspeech

`svgs` returns one SVG per sentence, `svg` exactly one and refuses a multi-sentence
text without `sentence` or `highlight`.

In [18]:
print(len(malt_ner.svgs(multi)), "SVGs")
try:
    malt_ner.svg(multi)
except ValueError as e:
    print("ValueError:", e)

6 SVGs
ValueError: the Text has 6 sentences; give `sentence` or `highlight`


## Tiny sentences

A one-word sentence has only the root arrow; a two-word sentence one arc.

In [19]:
hand(make_text("Tere", [([0], ["root"])]))
hand(make_text("Tere hommikust", [([2, 0], ["discourse", "root"])]))

root 
 Tere

root 
 discourse 
 Tere 
 hommikust

A highlight covering the whole sentence, and a window larger than the sentence.

In [20]:
small = make_text(
    "Ma lähen koju .", [([2, 0, 2, 2], ["nsubj", "root", "obl", "punct"])]
)
hand(small, highlight=small.words[:])
hand(small, highlight=[small.words[0]], window=100)

root 
 nsubj 
 obl 
 punct 
 Ma 
 lähen 
 koju 
 .

root 
 obl 
 punct 
 nsubj 
 Ma 
 lähen 
 koju 
 .

## Non-projective and unusual trees

Crossing arcs must stay on separate levels so their bars do not overlap, and labels
move off the legs that cross a bar. These trees are artificial.

In [21]:
crossing = make_text(
    "a b c d e f g h",
    [([3, 6, 0, 7, 3, 3, 8, 2], ["x1", "x2", "root", "x4", "x5", "x6", "x7", "x8"])],
)
hand(crossing)

root 
 x1 
 x2 
 x4 
 x5 
 x6 
 x7 
 x8 
 a 
 b 
 c 
 d 
 e 
 f 
 g 
 h

In [22]:
# Arcs that all cross each other: word i depends on word i + 4.
zigzag = make_text(
    "a b c d e f g h i j",
    [([5, 6, 7, 8, 9, 10, 0, 7, 7, 7], "dep")],
)
hand(zigzag)

dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 dep 
 a 
 b 
 c 
 d 
 e 
 f 
 g 
 h 
 i 
 j

Several roots in one sentence (parsers can produce this for fragments) and an arc
going across the root.

In [23]:
roots = make_text(
    "Pealkiri : uudised , ilm ja sport",
    [([0, 1, 0, 3, 3, 7, 3], ["root", "punct", "root", "punct", "conj", "cc", "conj"])],
)
hand(roots)

root 
 
 root 
 punct 
 punct 
 conj 
 cc 
 conj 
 Pealkiri 
 : 
 uudised 
 , 
 ilm 
 ja 
 sport

## Long labels, long words

A label longer than its short arc widens the gap between the words. Long words
make wide columns.

In [24]:
labels = make_text(
    "a b c d e",
    [
        (
            [2, 0, 2, 3, 4],
            [
                "a_very_long_dependency_relation",
                "root",
                "obl:tmod:extra",
                "nmod:poss:really:long",
                "x",
            ],
        )
    ],
)
hand(labels)

root 
 a_very_long_dependency_relation 
 obl:tmod:extra 
 nmod:poss:really:long 
 x 
 a 
 b 
 c 
 d 
 e

In [25]:
long_words = make_text(
    "Riigikogu põhiseaduskomisjoni esimees "
    "kommenteeris elektrijaotusvõrguettevõtete konkurentsivõimelisemaks muutmist",
    [
        (
            [2, 3, 4, 0, 6, 7, 4],
            ["nmod", "nmod", "nsubj", "root", "nmod", "xcomp", "obj"],
        )
    ],
)
hand(long_words)

root 
 nmod 
 nmod 
 nsubj 
 nmod 
 xcomp 
 obj 
 Riigikogu 
 põhiseaduskomisjoni 
 esimees 
 kommenteeris 
 elektrijaotusvõrguettevõtete 
 konkurentsivõimelisemaks 
 muutmist

Long row values (an ambiguous layer with `ambiguous="all"`) widen the columns too.

In [26]:
ambiguous = Text("Tema tuli ja läks.").tag_layer(["maltparser_syntax"])
DependencyTreeVisualiser(
    syntax_layer="maltparser_syntax",
    rows=[("morph_analysis", "partofspeech"), ("morph_analysis", "form")],
    ambiguous="all",
)(ambiguous)

root 
 nsubj 
 cc 
 conj 
 punct 
 Tema 
 P 
 sg n 
 tuli 
 V 
 s 
 ja 
 J 
 
 läks 
 V 
 s 
 . 
 Z 
 
 
 partofspeech 
 form

## Special characters

Characters with a meaning in XML (`<`, `>`, `&`, quotes) must be escaped, and
letters with diacritics must be measured close to their real width.

In [27]:
special = make_text(
    'x < y ja a & b , "tsitaat" ning <tag> Šveitsi žürii õõtsus',
    [
        (
            [4, 1, 1, 0, 4, 5, 5, 4, 10, 4, 10, 4, 4, 15, 4, 15],
            [
                "dep",
                "<lt>",
                "&amp;",
                "root",
                "dep",
                "a&b",
                "dep",
                "dep",
                '"q"',
                "dep",
                "'s'",
                "dep",
                "dep",
                "dep",
                "dep",
                "dep",
            ],
        )
    ],
)
svg = hand.svg(special)
hand(special)

root 
 dep 
 <lt> 
 &amp; 
 dep 
 a&b 
 dep 
 dep 
 "q" 
 dep 
 's' 
 dep 
 dep 
 dep 
 dep 
 dep 
 x 
 < 
 y 
 ja 
 a 
 & 
 b 
 , 
 " 
 tsitaat 
 " 
 ning 
 <tag> 
 Šveitsi 
 žürii 
 õõtsus

In [28]:
import xml.etree.ElementTree as ET

ET.fromstring(svg)  # raises if the SVG is not well-formed XML
print("well-formed,", len(svg), "characters")

well-formed, 7746 characters


## Boxes

Adjacent and nested spans of the span layer, spans at the sentence edges, and a
highlight over a box.

In [29]:
boxes = make_text(
    "Tartu Ülikooli rektor Toomas Asser ja Eesti Kunstiakadeemia esindaja kohtusid",
    [
        (
            [2, 3, 10, 3, 4, 9, 8, 9, 3, 0],
            [
                "nmod",
                "nmod",
                "nsubj",
                "appos",
                "flat",
                "cc",
                "nmod",
                "nmod",
                "conj",
                "root",
            ],
        )
    ],
)
entities = Layer(
    "entities", attributes=("label",), text_object=boxes, enveloping="words"
)
for start, end, label in [
    (0, 2, "ORG"),
    (2, 5, "PER+TITLE"),
    (3, 5, "PER"),
    (6, 8, "ORG"),
    (9, 10, "EVENT"),
]:
    entities.add_annotation(boxes.words[start:end], label=label)
boxes.add_layer(entities)

box_vis = DependencyTreeVisualiser(
    syntax_layer="syntax", span_layer="entities", span_label="label"
)
box_vis(boxes)

root 
 nmod 
 nmod 
 nsubj 
 appos 
 flat 
 cc 
 nmod 
 nmod 
 conj 
 Tartu 
 Ülikooli 
 rektor 
 Toomas 
 Asser 
 ja 
 Eesti 
 Kunstiakadeemia 
 esindaja 
 kohtusid 
 ORG 
 PER+TITLE 
 PER 
 ORG 
 EVENT

In [30]:
box_vis(boxes, highlight=boxes.entities[2], window=0)

appos 
 flat 
 … 
 rektor 
 Toomas 
 Asser 
 … 
 PER+TITLE 
 PER

Rows from a word-level layer that covers only some words: missing values are shown
as `-`, or any `missing` string.

In [31]:
marks = Layer("marks", attributes=("mark",), text_object=boxes)
for i in (0, 3, 4, 9):
    marks.add_annotation(boxes.words[i].base_span, mark=f"m{i}")
boxes.add_layer(marks)
DependencyTreeVisualiser(syntax_layer="syntax", rows=[("marks", "mark")], missing="·")(
    boxes
)

root 
 nmod 
 nmod 
 nsubj 
 appos 
 flat 
 cc 
 nmod 
 nmod 
 conj 
 Tartu 
 m0 
 Ülikooli 
 · 
 rektor 
 · 
 Toomas 
 m3 
 Asser 
 m4 
 ja 
 · 
 Eesti 
 · 
 Kunstiakadeemia 
 · 
 esindaja 
 · 
 kohtusid 
 m9 
 
 mark

## Extreme styles

A very compact style on the long sentence, and a large one on a short sentence.

In [32]:
tiny = Style(word_font_size=9, level_height=10, leg_gap=3)
DependencyTreeVisualiser(syntax_layer="maltparser_syntax", style=tiny)(long_text)

root 
 mark 
 nmod 
 nmod 
 nsubj 
 amod 
 nmod 
 obl 
 punct 
 mark 
 obl 
 cc 
 conj 
 amod 
 nmod 
 obl 
 ccomp 
 amod 
 acl 
 obj 
 punct 
 nmod 
 obj 
 acl:relcl 
 advmod 
 amod 
 obl 
 punct 
 aux 
 advcl 
 nsubj 
 xcomp 
 punct 
 mark 
 nmod 
 nsubj 
 ccomp 
 nmod 
 obl 
 punct 
 conj 
 cc 
 amod 
 conj 
 case 
 advmod 
 nummod 
 obl 
 xcomp 
 punct 
 mark 
 advmod 
 advcl 
 aux 
 punct 
 Kui 
 Eesti 
 Vabariigi 
 valitsus 
 otsustas 
 eelmise 
 aasta 
 sügisel 
 , 
 et 
 Tallinna 
 ja 
 Tartu 
 vahelise 
 raudtee 
 kiirendamiseks 
 ehitatakse 
 uus 
 elektrifitseeritud 
 rööbastee 
 , 
 mille 
 valmimist 
 oodati 
 juba 
 2028. 
 aastaks 
 , 
 ei 
 osanud 
 keegi 
 arvata 
 , 
 et 
 projekti 
 eelarve 
 kasvab 
 ehitusmaterjalide 
 hinnatõusu 
 , 
 tööjõupuuduse 
 ning 
 keeruliste 
 pinnasetingimuste 
 tõttu 
 peaaegu 
 kaks 
 korda 
 suuremaks 
 , 
 kui 
 algselt 
 planeeritud 
 oli 
 .

In [33]:
big = Style(word_font_size=24, level_height=36, leg_gap=8)
DependencyTreeVisualiser(syntax_layer="maltparser_syntax", style=big)(multi, sentence=1)

root 
 advmod 
 nsubj 
 advmod 
 obl 
 punct 
 Kas 
 sa 
 tulid 
 eile 
 koosolekule 
 ?

## Errors

Bad input gives a `ValueError` with a clear message.

In [34]:
cases = {
    "missing syntax layer": lambda: DependencyTreeVisualiser()(multi),
    "missing row layer": lambda: DependencyTreeVisualiser(
        syntax_layer="maltparser_syntax", rows=[("nope", "x")]
    )(multi),
    "sentence out of range": lambda: malt(multi, sentence=10),
    "highlight in another sentence": lambda: malt(
        multi, sentence=0, highlight=multi.words[-1]
    ),
    "indices without sentence": lambda: malt(multi, highlight=[0, 1]),
    "index outside the sentence": lambda: malt(multi, sentence=0, highlight=[50]),
    "empty highlight": lambda: malt(multi, highlight=[]),
    "negative window": lambda: malt(multi, highlight=multi.words[0], window=-1),
    "bad ambiguous": lambda: DependencyTreeVisualiser(ambiguous="some"),
}
for name, f in cases.items():
    try:
        f()
        print(f"{name}: no error")
    except ValueError as e:
        print(f"{name}: {e}")

missing syntax layer: Syntax layer 'stanza_syntax' is missing from the Text (layers: compound_tokens, maltparser_conll_morph, maltparser_syntax, morph_analysis, ner, sentences, tokens, words)
missing row layer: Row layer 'nope' is missing from the Text (layers: compound_tokens, maltparser_conll_morph, maltparser_syntax, morph_analysis, ner, sentences, tokens, words)
sentence out of range: sentence 10 is out of range (the Text has 6)
highlight in another sentence: the highlight is in sentence 5, not in 0
indices without sentence: give `sentence` when the highlight is word indices
index outside the sentence: highlight indices must be in 0..1
empty highlight: highlight is empty
negative window: window size must be >= 0, got -1
bad ambiguous: ambiguous must be 'first' or 'all', got 'some'
